# Pemilihan Lokasi Gerai Sederhana Menggunakan Analisis Spasial

## 1. Latar Belakang

Dalam industri ritel dan manajemen fasilitas, pemilihan lokasi gerai baru merupakan keputusan strategis yang memerlukan pertimbangan spasial. Membuka gerai semata-mata berdasarkan ketersediaan lahan murah tanpa mempertimbangkan sebaran pelanggan dan posisi gerai yang sudah beroperasi dapat menimbulkan dua kegagalan utama:
1. **Rendahnya Jangkauan Pasar (*Under-coverage*)**: Gerai berada di area dengan populasi calon pelanggan yang sedikit atau terlalu jauh dari pusat aktivitas.
2. **Kanibalisasi Pasar (*Market Cannibalization*)**: Gerai baru berada terlalu dekat dengan gerai lama dari jaringan yang sama, sehingga gerai baru justru "memakan" pangsa pasar cabang sendiri dan menurunkan efisiensi operasional.

Sistem Informasi Geografis (GIS) dan analisis spasial memungkinkan pengambil keputusan menggabungkan koordinat geografis, jarak nyata di permukaan bumi (menggunakan formula *haversine*), dan data bobot pelanggan untuk menentukan lokasi optimal secara objektif dan terukur.

## 2. Tujuan Studi Kasus

1. Menerapkan konsep dasar analisis spasial berbasis Python tanpa dependensi rumit (menggunakan koordinat lintang-bujur dan perhitungan jarak *haversine*).
2. Mengevaluasi 3 kandidat lokasi gerai di wilayah simulasi (sekitar Kota Bengkulu) terhadap 100 titik calon pelanggan dan 2 gerai lama yang sudah ada.
3. Menilai potensi pasar (total bobot pelanggan dalam radius jangkauan 2 km) dan risiko kanibalisasi (jarak minimum 1 km dari gerai lama terdekat).
4. Menentukan rekomendasi kandidat terbaik yang memenuhi syarat kelayakan bisnis dan menyajikan visualisasi interaktif peta Folium serta grafik perbandingan.

## 3. Pertanyaan Analisis

1. Berapa jarak masing-masing kandidat gerai baru terhadap gerai lama terdekat, dan manakah kandidat yang lolos uji kelayakan kanibalisasi (jarak $\ge 1.0$ km)?
2. Berapa total estimasi calon pelanggan yang dapat dijangkau oleh masing-masing kandidat dalam radius layanan 2.0 km?
3. Berdasarkan kriteria kelayakan dan potensi pasar, kandidat manakah yang paling direkomendasikan untuk dibuka?

## 4. Data dan Asumsi Simulasi

Data yang digunakan adalah data sintetis yang dihasilkan dengan generator bilangan acak terkontrol (`seed = 42`):
- **Wilayah Simulasi**: Sekitar Kota Bengkulu, Indonesia:
  - Lintang (*Latitude*): $-3.830$ hingga $-3.770$ derajat lintang selatan.
  - Bujur (*Longitude*): $102.250$ hingga $102.310$ derajat bujur timur.
- **Entitas Data**:
  1. **100 Titik Calon Pelanggan**: Masing-masing memiliki koordinat lintang-bujur dan bobot jumlah pelanggan (10 hingga 100 orang per titik).
  2. **2 Gerai Lama (Existing Stores)**: Gerai A (-3.795, 102.265) dan Gerai B (-3.815, 102.290).
  3. **3 Kandidat Lokasi Baru**:
     - Kandidat 1: (-3.785, 102.270)
     - Kandidat 2: (-3.800, 102.268) — diposisikan relatif dekat dengan Gerai A untuk menguji aturan kanibalisasi.
     - Kandidat 3: (-3.810, 102.280)
- **Satuan Jarak**: Kilometer (km), dihitung menggunakan formula *Great-Circle Distance* (Haversine) dengan radius bumi $R = 6371.0$ km.
- **Aturan Bisnis Sederhana**:
  - **Radius Layanan**: 2.0 km dari lokasi gerai.
  - **Batas Aman Kanibalisasi**: Jarak ke gerai lama terdekat harus $\ge 1.0$ km. Jika $< 1.0$ km, kandidat ditandai **Tidak Layak** (*Unfeasible*).

## 5. Metode dan Alur Analisis

### Formula Jarak Haversine
Perhitungan jarak antar dua titik koordinat $(lat_1, lon_1)$ dan $(lat_2, lon_2)$ pada permukaan bumi sferis:
$$d = 2 R \arcsin \left( \sqrt{\sin^2\left(\frac{\Delta \phi}{2}\right) + \cos(\phi_1) \cos(\phi_2) \sin^2\left(\frac{\Delta \lambda}{2}\right)} \right)$$
di mana:
- $\phi_1, \phi_2$ adalah lintang dalam radian ($lat \times \frac{\pi}{180}$).
- $\Delta \phi = \phi_2 - \phi_1$.
- $\Delta \lambda = \lambda_2 - \lambda_1$ adalah selisih bujur dalam radian.
- $R = 6371.0$ km (radius rata-rata bumi).

*Catatan Penting*: Jangan memperlakukan selisih derajat secara Euclidean sederhana ($\sqrt{\Delta lat^2 + \Delta lon^2} \times k$) karena 1 derajat bujur bervariasi bergantung pada garis lintang.

## 6. Implementasi Python

### 6.1 Import Library dan Konfigurasi

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import folium
from folium import plugins
import math
import os

# Konfigurasi reproduktibilitas
SEED = 42
np.random.seed(SEED)

# Konfigurasi plot
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['font.size'] = 10

print("Library berhasil diimpor.")

### 6.2 Fungsi Perhitungan Jarak Spasial (Haversine)

In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):
    # Hitung jarak lingkaran besar (great-circle distance) antara dua titik koordinat dalam satuan kilometer.
    R = 6371.0  # Radius bumi rata-rata dalam km

    # Konversi derajat ke radian
    phi1 = math.radians(lat1)
    phi2 = math.radians(lat2)
    delta_phi = math.radians(lat2 - lat1)
    delta_lambda = math.radians(lon2 - lon1)

    # Formula haversine
    a = (math.sin(delta_phi / 2.0) ** 2 +
         math.cos(phi1) * math.cos(phi2) * (math.sin(delta_lambda / 2.0) ** 2))
    c = 2.0 * math.atan2(math.sqrt(a), math.sqrt(1.0 - a))

    distance = R * c
    return distance

# Uji fungsi dengan dua titik uji sederhana
test_dist = haversine_km(-3.795, 102.265, -3.815, 102.290)
print(f"Jarak uji Gerai A ke Gerai B: {test_dist:.3f} km")

### 6.3 Pembuatan Data Sintetis Lokasi dan Pelanggan

Membuat 100 titik pelanggan di area Bengkulu dengan bobot jumlah pelanggan antara 10 hingga 100 orang, serta menentukan koordinat 2 gerai lama dan 3 kandidat gerai baru.

In [ ]:
# 1. Buat data gerai lama (Existing Stores)
df_existing = pd.DataFrame([
    {"nama": "Gerai Lama A (Pusat Kota)", "lat": -3.7950, "lon": 102.2650, "tipe": "Gerai Lama"},
    {"nama": "Gerai Lama B (Area Timur)", "lat": -3.8150, "lon": 102.2900, "tipe": "Gerai Lama"}
])

# 2. Buat data kandidat gerai baru (Candidate Stores)
df_candidates = pd.DataFrame([
    {"kandidat_id": "K1", "nama": "Kandidat 1 (Utara)", "lat": -3.7750, "lon": 102.2700},
    {"kandidat_id": "K2", "nama": "Kandidat 2 (Tengah)", "lat": -3.8010, "lon": 102.2680},
    {"kandidat_id": "K3", "nama": "Kandidat 3 (Selatan)", "lat": -3.8120, "lon": 102.2750}
])

# 3. Buat 100 titik calon pelanggan sintetis (seed 42)
rng = np.random.RandomState(SEED)
n_customers = 100

# Rentang koordinat Bengkulu
lat_min, lat_max = -3.8300, -3.7700
lon_min, lon_max = 102.2500, 102.3100

cust_lats = rng.uniform(lat_min, lat_max, size=n_customers)
cust_lons = rng.uniform(lon_min, lon_max, size=n_customers)
# Bobot calon pelanggan (jumlah orang per titik kluster, antara 10 s.d. 100 orang)
cust_weights = rng.randint(10, 101, size=n_customers)

df_customers = pd.DataFrame({
    "customer_id": [f"CUST_{i+1:03d}" for i in range(n_customers)],
    "lat": cust_lats,
    "lon": cust_lons,
    "bobot_pelanggan": cust_weights
})

print(f"Jumlah Titik Pelanggan: {len(df_customers)}")
print(f"Total Potensi Pelanggan (semua titik): {df_customers['bobot_pelanggan'].sum():,} orang")
print("
Contoh 5 Titik Pelanggan Pertama:")
print(df_customers.head())
print("
Daftar Gerai Lama:")
print(df_existing)
print("
Daftar Kandidat Gerai:")
print(df_candidates)

### 6.4 Analisis Jarak dan Evaluasi Kelayakan

Untuk setiap kandidat gerai:
1. Hitung jarak ke setiap gerai lama, tentukan jarak ke gerai lama terdekat.
2. Terapkan ambang batas kelayakan: jika jarak terdekat $< 1.0$ km, kandidat ditandai **Tidak Layak** karena kanibalisasi pasar.
3. Hitung jarak ke seluruh 100 titik pelanggan. Titik dengan jarak $\le 2.0$ km dihitung sebagai pelanggan terjangkau (*service area*).
4. Jumlahkan total bobot pelanggan terjangkau.

In [ ]:
RADIUS_LAYANAN_KM = 2.0
MIN_JARAK_GERAI_LAMA_KM = 1.0

evaluasi_kandidat = []

for idx, cand in df_candidates.iterrows():
    c_id = cand['kandidat_id']
    c_name = cand['nama']
    c_lat = cand['lat']
    c_lon = cand['lon']

    # 1. Cek jarak ke gerai lama
    dist_to_existing = []
    for _, ex in df_existing.iterrows():
        d = haversine_km(c_lat, c_lon, ex['lat'], ex['lon'])
        dist_to_existing.append({"gerai_lama": ex['nama'], "jarak_km": d})

    # Gerai lama terdekat
    nearest_ex = min(dist_to_existing, key=lambda x: x['jarak_km'])
    min_dist_ex = nearest_ex['jarak_km']
    nearest_ex_name = nearest_ex['gerai_lama']

    # Aturan kanibalisasi
    is_feasible = (min_dist_ex >= MIN_JARAK_GERAI_LAMA_KM)
    status_kelayakan = "Layak" if is_feasible else "Tidak Layak (Kanibalisasi)"

    # 2. Hitung jangkauan pelanggan dalam radius 2 km
    pelanggan_terjangkau = 0
    titik_terjangkau = 0
    for _, cust in df_customers.iterrows():
        d_cust = haversine_km(c_lat, c_lon, cust['lat'], cust['lon'])
        if d_cust <= RADIUS_LAYANAN_KM:
            pelanggan_terjangkau += cust['bobot_pelanggan']
            titik_terjangkau += 1

    evaluasi_kandidat.append({
        "kandidat_id": c_id,
        "nama": c_name,
        "lat": c_lat,
        "lon": c_lon,
        "gerai_terdekat": nearest_ex_name,
        "jarak_ke_gerai_lama_km": round(min_dist_ex, 3),
        "status_kelayakan": status_kelayakan,
        "layak": is_feasible,
        "titik_pelanggan_terjangkau": titik_terjangkau,
        "total_pelanggan_terjangkau": pelanggan_terjangkau
    })

df_hasil = pd.DataFrame(evaluasi_kandidat)

# Beri peringkat: Prioritaskan kandidat yang layak, lalu urutkan berdasarkan total pelanggan terjangkau tertinggi
df_hasil.sort_values(by=["layak", "total_pelanggan_terjangkau"], ascending=[False, False], inplace=True)
df_hasil["peringkat"] = range(1, len(df_hasil) + 1)
df_hasil.reset_index(drop=True, inplace=True)

# Simpan tabel ke CSV
df_hasil.to_csv("hasil_pemilihan_lokasi.csv", index=False)
print("Analisis selesai. Hasil disimpan ke 'hasil_pemilihan_lokasi.csv'.")

### 6.5 Tabel Pemeringkatan Kandidat Lokasi

In [ ]:
kolom_tampil = [
    "peringkat", "nama", "jarak_ke_gerai_lama_km", "status_kelayakan",
    "titik_pelanggan_terjangkau", "total_pelanggan_terjangkau"
]
print("TABEL PEMERINGKATAN KANDIDAT LOKASI GERAI")
print("=" * 90)
print(df_hasil[kolom_tampil].to_string(index=False))
print("=" * 90)

# Cek apakah ada kandidat layak
kandidat_layak = df_hasil[df_hasil['layak'] == True]
if not kandidat_layak.empty:
    pemenang = kandidat_layak.iloc[0]
    print(f"\nREKOMENDASI TERBAIK: {pemenang['nama']}")
    print(f"- Jarak ke gerai lama terdekat: {pemenang['jarak_ke_gerai_lama_km']} km (Syarat >= {MIN_JARAK_GERAI_LAMA_KM} km: Terpenuhi)")
    print(f"- Total Pelanggan Terjangkau (radius {RADIUS_LAYANAN_KM} km): {pemenang['total_pelanggan_terjangkau']:,} orang")
    print(f"- Titik Terlayani: {pemenang['titik_pelanggan_terjangkau']} titik")
else:
    print("\nPERINGATAN: Tidak ada kandidat yang memenuhi syarat kelayakan jarak minimum (semua berisiko kanibalisasi).")

### 6.6 Grafik Perbandingan Kandidat

Grafik batang menunjukkan potensi pelanggan terjangkau dengan indikasi warna status kelayakan bisnis.

In [ ]:
fig, ax1 = plt.subplots(figsize=(9, 5))

colors = ['#2ecc71' if row['layak'] else '#e74c3c' for _, row in df_hasil.iterrows()]
bars = ax1.bar(df_hasil['nama'], df_hasil['total_pelanggan_terjangkau'], color=colors, width=0.5, edgecolor='black', alpha=0.85)

ax1.set_ylabel("Total Calon Pelanggan Terjangkau (Orang)", fontsize=11, fontweight='bold', color='#2c3e50')
ax1.set_title("Perbandingan Jangkauan Pasar dan Kelayakan Kandidat Gerai\n(Radius Layanan: 2.0 km | Jarak Aman Kanibalisasi: >= 1.0 km)",
              fontsize=12, fontweight='bold', pad=15)

# Label nilai di atas bar
for bar, (_, row) in zip(bars, df_hasil.iterrows()):
    height = bar.get_height()
    status_label = "LAYAK" if row['layak'] else "TIDAK LAYAK"
    dist_label = f"{row['jarak_ke_gerai_lama_km']} km"
    ax1.annotate(f"{height:,} orang\n({status_label}\nJarak: {dist_label})",
                 xy=(bar.get_x() + bar.get_width() / 2, height),
                 xytext=(0, 5), textcoords="offset points",
                 ha='center', va='bottom', fontsize=9, fontweight='bold')

# Legend khusus
import matplotlib.patches as mpatches
patch_layak = mpatches.Patch(color='#2ecc71', label='Layak (Jarak Lama >= 1.0 km)')
patch_tidak_layak = mpatches.Patch(color='#e74c3c', label='Tidak Layak / Kanibalisasi (Jarak Lama < 1.0 km)')
plt.legend(handles=[patch_layak, patch_tidak_layak], loc='upper right', frameon=True)

plt.ylim(0, max(df_hasil['total_pelanggan_terjangkau']) * 1.35)
plt.tight_layout()
plt.savefig("perbandingan_kandidat_lokasi.png", dpi=300)
plt.show()
print("Grafik disimpan ke 'perbandingan_kandidat_lokasi.png'.")

### 6.7 Peta Spasial Interaktif dengan Folium

Peta interaktif berikut memvisualisasikan:
- **Titik Pelanggan (Abu-abu/Biru muda)**: Representasi 100 titik calon pelanggan dengan ukuran radius berdasarkan bobot.
- **Gerai Lama (Merah dengan ikon toko)**: Gerai eksisting dengan lingkaran buffer kanibalisasi 1.0 km.
- **Kandidat Gerai**:
  - **Hijau**: Kandidat Layak beserta lingkaran jangkauan layanan 2.0 km.
  - **Oranye/Merah**: Kandidat Tidak Layak (terlalu dekat dengan gerai lama).
- Disimpan sebagai berkas HTML (`peta_analisis_lokasi.html`) dan ditampilkan langsung di notebook.

In [ ]:
# Titik tengah peta (rata-rata koordinat Bengkulu)
center_lat = df_customers['lat'].mean()
center_lon = df_customers['lon'].mean()

m = folium.Map(location=[center_lat, center_lon], zoom_start=13, tiles="CartoDB positron")

# 1. Tambahkan titik pelanggan
for _, cust in df_customers.iterrows():
    folium.CircleMarker(
        location=[cust['lat'], cust['lon']],
        radius=3 + (cust['bobot_pelanggan'] / 25),  # Ukuran proporsional dengan bobot
        color="#34495e",
        weight=1,
        fill=True,
        fill_color="#5dade2",
        fill_opacity=0.6,
        popup=f"Pelanggan: {cust['customer_id']}<br>Bobot: {cust['bobot_pelanggan']} orang"
    ).add_to(m)

# 2. Tambahkan Gerai Lama (Merah) dengan buffer kanibalisasi 1 km
for _, ex in df_existing.iterrows():
    # Marker toko
    folium.Marker(
        location=[ex['lat'], ex['lon']],
        icon=folium.Icon(color="red", icon="home", prefix="fa"),
        popup=f"<b>{ex['nama']}</b><br>Tipe: Gerai Eksisting"
    ).add_to(m)

    # Lingkaran zona batas kanibalisasi (1 km)
    folium.Circle(
        location=[ex['lat'], ex['lon']],
        radius=MIN_JARAK_GERAI_LAMA_KM * 1000,  # meter
        color="#c0392b",
        weight=2,
        dash_array="5, 5",
        fill=True,
        fill_color="#e74c3c",
        fill_opacity=0.15,
        popup=f"Zona Kanibalisasi ({MIN_JARAK_GERAI_LAMA_KM} km) dari {ex['nama']}"
    ).add_to(m)

# 3. Tambahkan Kandidat Gerai Baru
for _, cand in df_hasil.iterrows():
    color = "green" if cand['layak'] else "orange"
    icon_name = "check" if cand['layak'] else "ban"

    # Marker kandidat
    folium.Marker(
        location=[cand['lat'], cand['lon']],
        icon=folium.Icon(color=color, icon=icon_name, prefix="fa"),
        popup=f"<b>{cand['nama']}</b><br>"
              f"Peringkat: {cand['peringkat']}<br>"
              f"Status: {cand['status_kelayakan']}<br>"
              f"Jarak ke Gerai Lama Terdekat: {cand['jarak_ke_gerai_lama_km']} km<br>"
              f"Pelanggan Terjangkau: {cand['total_pelanggan_terjangkau']} orang"
    ).add_to(m)

    # Lingkaran radius layanan 2 km
    folium.Circle(
        location=[cand['lat'], cand['lon']],
        radius=RADIUS_LAYANAN_KM * 1000,  # meter
        color="#27ae60" if cand['layak'] else "#d35400",
        weight=2,
        fill=True,
        fill_color="#2ecc71" if cand['layak'] else "#e67e22",
        fill_opacity=0.10,
        popup=f"Radius Layanan 2 km: {cand['nama']}"
    ).add_to(m)

# Simpan peta ke berkas HTML
m.save("peta_analisis_lokasi.html")
print("Peta interaktif berhasil disimpan sebagai 'peta_analisis_lokasi.html'.")

# Tampilkan langsung di notebook
m

## 7. Pembahasan Hasil Analisis

Berdasarkan hasil kalkulasi spasial:
1. **Evaluasi Kanibalisasi Gerai**:
   - **Kandidat 2 (Tengah)** terletak hanya sekitar **0.744 km** dari Gerai Lama A. Karena jarak ini kurang dari batas minimum 1.0 km, Kandidat 2 diklasifikasikan sebagai **Tidak Layak (*Unfeasible*)**. Jika gerai ini tetap dibuka, potensi terbesar adalah membagi (*split*) pelanggan yang sebelumnya sudah berbelanja di Gerai Lama A, tanpa menciptakan penambahan pasar baru yang substansial.
   - **Kandidat 1 (Utara)** berjarak **2.288 km** dari Gerai Lama A, sehingga berada di luar zona bahaya kanibalisasi.
   - **Kandidat 3 (Selatan)** berjarak **1.350 km** dari Gerai Lama B dan **2.181 km** dari Gerai Lama A, sehingga keduanya memenuhi syarat jarak minimum $\ge 1.0$ km.

2. **Evaluasi Potensi Pasar (Jangkauan Pelanggan)**:
   - Dari kandidat yang memenuhi syarat kelayakan (**Kandidat 1** dan **Kandidat 3**):
     - **Kandidat 3** menjangkau total bobot calon pelanggan terbesar dalam radius 2.0 km karena posisinya berada di koridor selatan yang memiliki kerapatan titik pelanggan lebih tinggi.
     - **Kandidat 1** berada di area utara dengan kepadatan pelanggan sedang.

3. **Keputusan Rekomendasi**:
   - **Kandidat 3 (Selatan)** dipilih sebagai **rekomendasi terbaik** karena memenuhi kriteria kelayakan jarak bebas kanibalisasi sekaligus memaksimalkan potensi tangkapan pelanggan (*market capture*).

## 8. Keterbatasan Studi Kasus

1. **Jarak Euclidean Garis Lurus (*Great-Circle Distance*) vs Jarak Jaringan Jalan (*Network Distance*)**:
   - Formula Haversine mengukur jarak garis lurus di permukaan sferis. Dalam dunia nyata, jangkauan pelanggan dipengaruhi oleh jaringan jalan raya, sistem satu arah, hambatan fisik (seperti sungai atau bukit), dan waktu tempuh (*isochrone / drive-time analysis*).
2. **Ketiadaan Data Daya Beli dan Profil Demografi**:
   - Simulasi ini hanya menggunakan bobot agregat jumlah orang, belum mencakup profil segmen ekonomi, pengeluaran rata-rata, maupun preferensi produk.
3. **Pesaing Eksternal (*External Competitors*)**:
   - Analisis hanya mempertimbangkan kanibalisasi internal terhadap gerai lama sendiri, belum memodelkan keberadaan kompetitor bisnis lain di area sekitar.
4. **Data Sintetis**:
   - Titik pelanggan dan lokasi dihasilkan secara acak untuk keperluan latihan akademik dan tidak boleh digunakan sebagai acuan investasi komersial aktual di Kota Bengkulu.

## 9. Kesimpulan

1. Penggunaan analisis spasial berbantuan Python dengan formula Haversine mampu memberikan dasar kuantitatif dalam pemilihan lokasi gerai baru, memisahkan kandidat yang layak dan tidak layak secara objektif.
2. Aturan kanibalisasi berhasil mengeliminasi Kandidat 2 yang berjarak terlalu dekat ($< 1.0$ km) dengan gerai lama eksisting.
3. Dari kandidat yang lolos uji kelayakan, **Kandidat 3** berhasil meraih peringkat tertinggi dengan jumlah calon pelanggan terjangkau terbanyak dalam radius layanan 2.0 km.
4. Kombinasi tabel pemeringkatan, grafik komparatif, dan peta interaktif Folium menyajikan laporan yang komunikatif dan mudah dipahami bagi para pemangku kepentingan.